In [2]:

from google.colab import files

archivos = files.upload()


Saving M20.csv to M20 (1).csv


In [3]:

import pandas as pd

datos = pd.read_csv("M20.csv")

display(datos)


,caso,conjunto,Disparos a la cabeza (% de los impactos),Tiempo de reacción (ms),Horas en la cuenta (h totales),resultado_real
0,1,entrenamiento,58,398,1960,0
1,2,entrenamiento,20,125,772,0
2,3,entrenamiento,18,355,1777,0
3,4,entrenamiento,79,311,607,0
4,5,entrenamiento,64,329,1358,0
5,6,entrenamiento,61,282,1849,0
6,7,entrenamiento,83,123,1988,0
7,8,entrenamiento,36,123,1159,0
8,9,entrenamiento,33,150,1398,0
9,10,entrenamiento,88,160,254,1


In [4]:

# Separar los dos conjuntos
entrenamiento = datos[datos["conjunto"] == "entrenamiento"]
prueba = datos[datos["conjunto"] == "prueba"]

# Las tres características de cada jugador
variables = [
    "Disparos a la cabeza (% de los impactos)",
    "Tiempo de reacción (ms)",
    "Horas en la cuenta (h totales)"
]

X_entrenamiento = entrenamiento[variables]
y_entrenamiento = entrenamiento["resultado_real"]

X_prueba = prueba[variables]
y_prueba = prueba["resultado_real"]

print("Casos de entrenamiento:", len(entrenamiento))
print("Casos de prueba:", len(prueba))


Casos de entrenamiento: 14
Casos de prueba: 6


In [5]:

from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Crear el modelo
modelo = make_pipeline(
    StandardScaler(),
    Perceptron(random_state=42, max_iter=10000)
)

# Entrenar usando SOLO los primeros 14 casos
modelo.fit(X_entrenamiento, y_entrenamiento)

print("Perceptrón entrenado correctamente")


Perceptrón entrenado correctamente


In [6]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

def evaluar(nombre, X, y):
    predicciones = modelo.predict(X)

    tn, fp, fn, tp = confusion_matrix(
        y, predicciones, labels=[0, 1]
    ).ravel()

    exactitud = accuracy_score(y, predicciones)
    precision = precision_score(y, predicciones, zero_division=0)
    sensibilidad = recall_score(y, predicciones, zero_division=0)
    especificidad = tn / (tn + fp) if tn + fp else 0

    print("\n", nombre)
    print("Matriz de confusión:")
    print([[tn, fp], [fn, tp]])
    print("Exactitud:", round(exactitud, 3))
    print("Precisión:", round(precision, 3))
    print("Sensibilidad:", round(sensibilidad, 3))
    print("Especificidad:", round(especificidad, 3))

evaluar("ENTRENAMIENTO", X_entrenamiento, y_entrenamiento)
evaluar("PRUEBA", X_prueba, y_prueba)



 ENTRENAMIENTO
Matriz de confusión:
[[np.int64(12), np.int64(0)], [np.int64(0), np.int64(2)]]
Exactitud: 1.0
Precisión: 1.0
Sensibilidad: 1.0
Especificidad: 1.0

 PRUEBA
Matriz de confusión:
[[np.int64(5), np.int64(0)], [np.int64(0), np.int64(1)]]
Exactitud: 1.0
Precisión: 1.0
Sensibilidad: 1.0
Especificidad: 1.0


In [8]:


def analizar_jugador(cabeza, reaccion, horas):
    jugador = pd.DataFrame([{
        variables[0]: cabeza,
        variables[1]: reaccion,
        variables[2]: horas
    }])

    resultado = modelo.predict(jugador)[0]

    if resultado == 1:
        print("SÍ: posible uso de aimbot.")
        print("Enviar a revisión manual.")
    else:
        print("NO: no se detectan indicios suficientes.")

# Ejemplo de jugador nuevo
analizar_jugador(85, 140, 200)


SÍ: posible uso de aimbot.
Enviar a revisión manual.


In [9]:

# Sistema básico: siempre predice NO
prediccion_basica = [0] * len(y_prueba)

# Predicciones del perceptrón
prediccion_ia = modelo.predict(X_prueba)

print("Exactitud sistema básico:",
      accuracy_score(y_prueba, prediccion_basica))

print("Exactitud perceptrón:",
      accuracy_score(y_prueba, prediccion_ia))


Exactitud sistema básico: 0.8333333333333334
Exactitud perceptrón: 1.0
